In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb

In [3]:
# -------------------------------
# 1. Data Loading & Decoding
# -------------------------------
max_features = 10000  # Vocabulary size
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)

print(f'Training data shape: {X_train.shape}, Training labels shape: {y_train.shape}')
print(f'Testing data shape: {X_test.shape}, Testing labels shape: {y_test.shape}')

Training data shape: (25000,), Training labels shape: (25000,)
Testing data shape: (25000,), Testing labels shape: (25000,)


In [4]:
# Obtain the word index for decoding and shift by 3 as per IMDB convention.
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

# Function to decode a single review into words.
def decode_review(review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in review])

# Decode a sample review for verification.
sample_review = X_train[0]
print("Sample review (decoded):")
print(decode_review(sample_review))

# For BERT, we need raw text. Convert the integer-encoded reviews to text.
num_train_samples = 2000  # Adjust as per available resources
num_test_samples = 500

X_train_text = [decode_review(review) for review in X_train[:num_train_samples]]
y_train_small = y_train[:num_train_samples]

X_test_text = [decode_review(review) for review in X_test[:num_test_samples]]
y_test_small = y_test[:num_test_samples]

Sample review (decoded):
? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what the

In [5]:
# -------------------------------
# 2. BERT Tokenization & Encoding
# -------------------------------

from transformers import BertTokenizer, TFBertForSequenceClassification

# Load pre-trained BERT tokenizer. (Using 'bert-base-uncased')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

max_length = 128  # Maximum sequence length for BERT

# Tokenize the text data.
def encode_texts(texts, max_length):
    return tokenizer(texts,
                     padding='max_length',
                     truncation=True,
                     max_length=max_length,
                     return_tensors='tf')

# Encode training and testing texts.
train_encodings = encode_texts(X_train_text, max_length)
test_encodings = encode_texts(X_test_text, max_length)

d:\simple_rnn_imdb\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\simple_rnn_imdb\env\lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\npi12\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: h

In [6]:
# -------------------------------
# 3. Prepare TensorFlow Datasets
# -------------------------------
# Create TensorFlow Dataset objects for training and validation.
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    y_train_small
)).shuffle(1000).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    y_test_small
)).batch(16)

In [7]:
# -------------------------------
# 4. Load & Fine-tune Pre-trained BERT
# -------------------------------
# Load BERT for sequence classification with 2 output labels.
model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Compile the model with an appropriate optimizer, loss, and metrics.
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-08)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')

model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

model.summary()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: "tf_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  109482240 
                                                                 
 dropout_37 (Dropout)        multiple                  0 (unused)
                                                                 
 classifier (Dense)          multiple                  1538      
                                                                 
Total params: 109483778 (417.65 MB)
Trainable params: 109483778 (417.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
# -------------------------------
# 5. Fine-tune the Model
# -------------------------------
# Use early stopping and model checkpoint callbacks as desired.
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

earlystopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
# checkpoint = ModelCheckpoint('bert_imdb_best.h5', monitor='val_loss', save_best_only=True, verbose=1)
checkpoint = ModelCheckpoint(
    'bert_imdb_best.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1,
    save_weights_only=True  # Only weights are saved
)


# Fine-tune the model. (Adjust epochs and steps as needed.)
history = model.fit(train_dataset,
                    validation_data=test_dataset,
                    callbacks=[earlystopping, checkpoint])

Epoch 1/3

125/125 [==============================] - ETA: 0s - loss: 0.5445 - accuracy: 0.7250
Epoch 1: val_loss improved from inf to 0.34940, saving model to bert_imdb_best.h5


d:\simple_rnn_imdb\env\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


NotImplementedError: Saving the model to HDF5 format requires the model to be a Functional model or a Sequential model. It does not work for subclassed models, because such models are defined via the body of a Python method, which isn't safely serializable. Consider saving to the Tensorflow SavedModel format (by setting save_format="tf") or using `save_weights`.

In [ ]:
# -------------------------------
# 6. Save the Final Model
# -------------------------------
model.save_pretrained("bert_imdb_final")



Training data shape: (25000,), Training labels shape: (25000,)
Testing data shape: (25000,), Testing labels shape: (25000,)
Sample review (decoded):
? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play 

ModuleNotFoundError: No module named 'transformers'